In [1]:
!pip install pandapower
!pip install pypsa

  Using cached pypsa-0.33.1-py3-none-any.whl.metadata (12 kB)
  Using cached netCDF4-1.7.2-cp312-cp312-win_amd64.whl.metadata (1.8 kB)
  Using cached linopy-0.5.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached geopandas-1.0.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached deprecation-2.1.0-py2.py3-none-any.whl.metadata (4.6 kB)
  Using cached validators-0.34.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached highspy-1.9.0-cp312-cp312-win_amd64.whl.metadata (10 kB)
  Using cached pyogrio-0.10.0-cp312-cp312-win_amd64.whl.metadata (5.6 kB)
  Using cached pyproj-3.7.1-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached shapely-2.0.7-cp312-cp312-win_amd64.whl.metadata (7.1 kB)
  Using cached xarray-2025.1.2-py3-none-any.whl.metadata (11 kB)
  Using cached polars-1.24.0-cp39-abi3-win_amd64.whl.metadata (15 kB)
  Using cached cftime-1.6.4.post1-cp312-cp312-win_amd64.whl.metadata (8.9 kB)
Using cached pypsa-0.33.1-py3-none-any.whl (173 kB)
Using cached geopandas-1.0.1-py3-none-any

In [11]:
import pandapower as pp

# Create an empty network
net = pp.create_empty_network()

# Add Buses
b1 = pp.create_bus(net, vn_kv=230, name="Bus 1")  # Slack Bus
b2 = pp.create_bus(net, vn_kv=230, name="Bus 2")
b3 = pp.create_bus(net, vn_kv=230, name="Bus 3")
b4 = pp.create_bus(net, vn_kv=230, name="Bus 4")
b5 = pp.create_bus(net, vn_kv=230, name="Bus 5")

# Add Slack Bus (Reference Bus)
pp.create_ext_grid(net, bus=b1, vm_pu=1.06, name="Slack")

# Add Generators (PV Bus)
pp.create_gen(net, bus=b3, p_mw=40, vm_pu=1.04, name="Gen at Bus 3")

# Add Loads (PQ Buses)
pp.create_load(net, bus=b2, p_mw=50, q_mvar=30, name="Load at Bus 2")
pp.create_load(net, bus=b4, p_mw=40, q_mvar=20, name="Load at Bus 4")
pp.create_load(net, bus=b5, p_mw=50, q_mvar=10, name="Load at Bus 5")

# Add Transmission Lines with max_i_ka (maximum current in kA)
pp.create_line_from_parameters(net, from_bus=b1, to_bus=b2, length_km=47, r_ohm_per_km=0.000426, x_ohm_per_km=0.0013, c_nf_per_km=30, max_i_ka=1.0)
pp.create_line_from_parameters(net, from_bus=b1, to_bus=b3, length_km=47, r_ohm_per_km=0.0017, x_ohm_per_km=0.0051, c_nf_per_km=25, max_i_ka=1.0)
pp.create_line_from_parameters(net, from_bus=b2, to_bus=b3, length_km=47, r_ohm_per_km=0.0013, x_ohm_per_km=0.0038, c_nf_per_km=20, max_i_ka=1.0)
pp.create_line_from_parameters(net, from_bus=b2, to_bus=b4, length_km=47, r_ohm_per_km=0.0013, x_ohm_per_km=0.0038, c_nf_per_km=20, max_i_ka=1.0)
pp.create_line_from_parameters(net, from_bus=b2, to_bus=b5, length_km=47, r_ohm_per_km=0.00085, x_ohm_per_km=0.00255, c_nf_per_km=15, max_i_ka=1.0)
pp.create_line_from_parameters(net, from_bus=b3, to_bus=b4, length_km=47, r_ohm_per_km=0.00021, x_ohm_per_km=0.00064, c_nf_per_km=10, max_i_ka=1.0)
pp.create_line_from_parameters(net, from_bus=b4, to_bus=b5, length_km=47, r_ohm_per_km=0.0017, x_ohm_per_km=0.0051, c_nf_per_km=25, max_i_ka=1.0)

# Run Power Flow Analysis
pp.runpp(net)

# Print Results
print("Bus Voltage Results:")
print(net.res_bus)

print("\nLine Loading Results:")
print(net.res_line)

Bus Voltage Results:
      vm_pu  va_degree        p_mw        q_mvar
0  1.060000   0.000000 -178.737595 -12491.096872
1  1.051405   0.144653   50.000000     30.000000
2  1.040000   0.356233  -40.000000  12319.894483
3  1.042289   0.312980   40.000000     20.000000
4  1.048352   0.196312   50.000000     10.000000

Line Loading Results:
    p_from_mw  q_from_mvar     p_to_mw    q_to_mvar      pl_mw    ql_mvar  \
0  135.099695  7833.229223 -114.354908 -7796.040043  20.744787  37.189180   
1   43.637900  4657.867649  -14.333357 -4591.484882  29.304543  66.382767   
2   17.790478  3539.350566   -4.637636 -3517.986742  13.152843  21.363824   
3   16.191163  2826.110953   -7.794873 -2818.688135   8.396290   7.422819   
4   30.373267  1400.578524  -29.020120 -1409.433474   1.353147  -8.854950   
5   58.970993 -4210.422859  -55.918464  4211.258870   3.052529   0.836011   
6   23.713337 -1412.570735  -20.979880  1399.433474   2.733457 -13.137261   

   i_from_ka    i_to_ka       i_ka  vm_from_p

In [5]:
import pypsa  # Import the PyPSA library for power system analysis [1, 2, 3]

# Define the power system network
network = pypsa.Network()  

# Add buses (nodes) to the network
network.add("bus", "bus1",  bus_type="slack", v_nom=1.05)  # Slack bus
network.add("bus", "bus2", bus_type="pv",v_nom=1.0) 
network.add("bus", "bus3", bus_type="pq",v_nom=1.0) 
network.add("bus", "bus4", bus_type="pq",v_nom=1.0) 
network.add("bus", "bus5", bus_type="pq",v_nom=1.0) 

# Add lines (branches) between buses
network.add("line", "line1",  from_bus="bus1", to_bus="bus2", r=0.02, x=0.06) 
network.add("line", "line2", from_bus="bus1", to_bus="bus3", r=0.08, x=0.24)
network.add("line", "line3", from_bus="bus2", to_bus="bus3", r=0.06, x=0.18)
network.add("line", "line4", from_bus="bus2", to_bus="bus4", r=0.06, x=0.18)
network.add("line", "line5", from_bus="bus2", to_bus="bus5", r=0.04, x=0.12)
network.add("line", "line6", from_bus="bus3", to_bus="bus4", r=0.01, x=0.03)
network.add("line", "line7", from_bus="bus4", to_bus="bus5", r=0.08, x=0.24)


# Add loads at specific buses
network.add("load", "load1", bus="bus2", p=29, q=0)
network.add("load", "load2", bus="bus3", p=84, q=0) 
network.add("load", "load3", bus="bus4", p=45, q=0)
network.add("load", "load4", bus="bus5", p=45, q=0) 


# Solve the load flow analysis 
network.solve_powerflow()  

# Access results
print("Voltage at bus 2:", network.buses_t.v_mag_pu["bus2"])  
print("Power flow on line 1:", network.lines_t.p_from["line1"])






C:\Users\pravh\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


AttributeError: Network has no components 'bus'

In [23]:
import pandapower as pp
import pandas as pd

# Create an empty power network
net = pp.create_empty_network()

# Add Buses
b1 = pp.create_bus(net, vn_kv=100, name="Bus 1")  # Slack Bus
b2 = pp.create_bus(net, vn_kv=100, name="Bus 2")
b3 = pp.create_bus(net, vn_kv=100, name="Bus 3")
b4 = pp.create_bus(net, vn_kv=100, name="Bus 4")
b5 = pp.create_bus(net, vn_kv=100, name="Bus 5")

# Add Slack Bus (Reference Bus)
pp.create_ext_grid(net, bus=b1, vm_pu=1.05, name="Slack")

# Add Generators (PV Bus)
pp.create_gen(net, bus=b2, p_mw=29, vm_pu=1, name="Gen at Bus 2")

# Add Loads (PQ Buses)
pp.create_load(net, bus=b3, p_mw=84, q_mvar=0, name="Load at Bus 3")
pp.create_load(net, bus=b4, p_mw=45, q_mvar=0, name="Load at Bus 4")
pp.create_load(net, bus=b5, p_mw=45, q_mvar=0, name="Load at Bus 5")

# Add Transmission Lines with updated parameters
pp.create_line_from_parameters(net, from_bus=b1, to_bus=b2, length_km=47, r_ohm_per_km=0.000426, x_ohm_per_km=0.0013, c_nf_per_km=30, max_i_ka=1.0)
pp.create_line_from_parameters(net, from_bus=b1, to_bus=b3, length_km=47, r_ohm_per_km=0.0017, x_ohm_per_km=0.0051, c_nf_per_km=25, max_i_ka=1.0)
pp.create_line_from_parameters(net, from_bus=b2, to_bus=b3, length_km=47, r_ohm_per_km=0.0013, x_ohm_per_km=0.0038, c_nf_per_km=20, max_i_ka=1.0)
pp.create_line_from_parameters(net, from_bus=b2, to_bus=b4, length_km=47, r_ohm_per_km=0.0013, x_ohm_per_km=0.0038, c_nf_per_km=20, max_i_ka=1.0)
pp.create_line_from_parameters(net, from_bus=b2, to_bus=b5, length_km=47, r_ohm_per_km=0.00085, x_ohm_per_km=0.00255, c_nf_per_km=15, max_i_ka=1.0)
pp.create_line_from_parameters(net, from_bus=b3, to_bus=b4, length_km=47, r_ohm_per_km=0.00021, x_ohm_per_km=0.00064, c_nf_per_km=10, max_i_ka=1.0)
pp.create_line_from_parameters(net, from_bus=b4, to_bus=b5, length_km=47, r_ohm_per_km=0.0017, x_ohm_per_km=0.0051, c_nf_per_km=25, max_i_ka=1.0)
# pp.create_line_from_parameters(net, from_bus=b1, to_bus=b2, length_km=1, r_ohm_per_km=0.02, x_ohm_per_km=0.06, c_nf_per_km=30, max_i_ka=1.0)
# pp.create_line_from_parameters(net, from_bus=b1, to_bus=b3, length_km=1, r_ohm_per_km=0.08, x_ohm_per_km=0.24, c_nf_per_km=25, max_i_ka=1.0)
# pp.create_line_from_parameters(net, from_bus=b2, to_bus=b3, length_km=1, r_ohm_per_km=0.06, x_ohm_per_km=0.18, c_nf_per_km=20, max_i_ka=1.0)
# pp.create_line_from_parameters(net, from_bus=b2, to_bus=b4, length_km=1, r_ohm_per_km=0.06, x_ohm_per_km=0.18, c_nf_per_km=20, max_i_ka=1.0)
# pp.create_line_from_parameters(net, from_bus=b2, to_bus=b5, length_km=1, r_ohm_per_km=0.04, x_ohm_per_km=0.12, c_nf_per_km=15, max_i_ka=1.0)
# pp.create_line_from_parameters(net, from_bus=b3, to_bus=b4, length_km=1, r_ohm_per_km=0.01, x_ohm_per_km=0.03, c_nf_per_km=10, max_i_ka=1.0)
# pp.create_line_from_parameters(net, from_bus=b4, to_bus=b5, length_km=1, r_ohm_per_km=0.08, x_ohm_per_km=0.24, c_nf_per_km=25, max_i_ka=1.0)

# Run Power Flow Analysis
pp.runpp(net)

# Extract line flows in MW
line_flows = net.res_line["p_from_mw"].tolist()

# Format output as a table
output_df = pd.DataFrame({
    "Line": ["1-2", "1-3", "2-3", "2-4", "2-5", "3-4", "4-5"],
    "MW Flow": line_flows
})

# Print in the required format
print("Table: Line Flows During Base Case\n")
print(output_df.to_string(index=False))
print("Bus Voltage Results:")
print(net.res_bus)

print("\nLine Loading Results:")
print(net.res_line)

Table: Line Flows During Base Case

Line    MW Flow
 1-2 216.873972
 1-3  86.401432
 2-3  30.748799
 2-4  33.673751
 2-5  48.893444
 3-4  11.044271
 4-5  -2.936296
Bus Voltage Results:
      vm_pu  va_degree        p_mw        q_mvar
0  1.050000   0.000000 -303.275404 -10155.579890
1  1.000000   0.860849  -29.000000   9696.287143
2  1.012430   0.585704   84.000000      0.000000
3  1.009890   0.630938   45.000000      0.000000
4  1.003199   0.762790   45.000000      0.000000

Line Loading Results:
    p_from_mw  q_from_mvar    p_to_mw    q_to_mvar       pl_mw     ql_mvar  \
0  216.873972  8538.357874 -84.315994 -8138.494886  132.557978  399.862988   
1   86.401432  1617.222017 -67.345314 -1563.980387   19.056118   53.241630   
2   30.748799  -707.316670 -27.698957   713.241560    3.049842    5.924889   
3   33.673751  -566.281412 -31.717705   569.016637    1.956046    2.735226   
4   48.893444  -284.194175 -48.563742   282.961362    0.329702   -1.232813   
5   11.044271   850.738827 -10